#### **1. FastQC & Falco**
To perform high throughput quality check
| Summary | Discription |
| --- | --- |
| Basic Statistics | Total sequences, total bases (per sequence), sequence length (eg. 150 or 200), %GC |
| Per base sequence quality | Distribution of base quality (shown in boxplot per 5 bases) across the sequence where 40-28 (green), 28-20 (yellow), 20-0 (red) |
| Per sequence quality scores | Line graph of base quality over the number of sequence |
| Per base sequence content | Content percentage of each type of base (A-G-T-C) per 5 bases across the sequence |
| Per sequence GC content | Distribution of GC content percentage of the sequences |
| Per base N content | Content percentage of unknown base per 5 bases across the sequence |
| Sequence Length Distribution | Distribution of length (bp) of the sequences |
| Sequence Duplication Levels | Percentage of sequnce duplication over duplication rate |
| Overrepresented sequences | Overreprented sequences |
| Adapter Content | Percentage of adapter content across the sequence |

In [2]:
from pathlib import Path
import subprocess

input_file = Path("/Users/nguyuling/Galaxy/datasets/582600/mutant_R1.fastq")
output_dir = Path("/Users/nguyuling/Galaxy/intro/fastqc_results")
output_dir.mkdir(exist_ok=True)

if not input_file.exists():
    raise FileNotFoundError(f"Input file not found: {input_file}")

subprocess.run(
    ["fastqc", str(input_file), "--outdir", str(output_dir)],
    check=True
)

print(f"FastQC results saved to: {output_dir}")

null


Started analysis of mutant_R1.fastq
Approx 5% complete for mutant_R1.fastq
Approx 15% complete for mutant_R1.fastq
Approx 20% complete for mutant_R1.fastq
Approx 30% complete for mutant_R1.fastq
Approx 40% complete for mutant_R1.fastq
Approx 45% complete for mutant_R1.fastq
Approx 55% complete for mutant_R1.fastq
Approx 60% complete for mutant_R1.fastq
Approx 70% complete for mutant_R1.fastq
Approx 80% complete for mutant_R1.fastq
Approx 85% complete for mutant_R1.fastq
Approx 95% complete for mutant_R1.fastq


Analysis complete for mutant_R1.fastq
FastQC results saved to: /Users/nguyuling/Galaxy/intro/fastqc_results


In [3]:
from pathlib import Path
import subprocess

input_file = Path("/Users/nguyuling/Galaxy/datasets/582600/mutant_R1.fastq")
output_dir = Path("/Users/nguyuling/Galaxy/intro/falco_results")
output_dir.mkdir(parents=True, exist_ok=True)

if not input_file.exists():
    raise FileNotFoundError(f"Input file not found: {input_file}")

subprocess.run(
    [
        "conda", "run", "--no-capture-output", "-n", "falco",
        "falco", "-o", str(output_dir), str(input_file)
    ],
    check=True
)

print(f"Falco results saved to: {output_dir}")

Falco results saved to: /Users/nguyuling/Galaxy/intro/falco_results


[limits]	WARNING: using default limits because limits file does not exist: ./Configuration/limits.txt
[adapters]	WARNING: using default adapters because adapters file does not exist: ./Configuration/adapter_list.txt
[contaminants]	WARNING: using default contaminants because contaminants file does not exist: ./Configuration/contaminant_list.txt
[Thu Sep 10 14:35:17 2026] Started reading file /Users/nguyuling/Galaxy/datasets/582600/mutant_R1.fastq
[Thu Sep 10 14:35:17 2026] reading file as uncompressed FASTQ format
[Thu Sep 10 14:35:17 2026] Finished reading file
[Thu Sep 10 14:35:17 2026] Writing summary to /Users/nguyuling/Galaxy/intro/falco_results/summary.txt
[Thu Sep 10 14:35:17 2026] Writing text report to /Users/nguyuling/Galaxy/intro/falco_results/fastqc_data.txt
[Thu Sep 10 14:35:17 2026] Writing HTML report to /Users/nguyuling/Galaxy/intro/falco_results/fastqc_report.html
Elapsed time for file /Users/nguyuling/Galaxy/datasets/582600/mutant_R1.fastq: 0s


#### **2. Filter by quality**
To filter low quality reads
| Parameter | Discription |
| --- | --- |
| Quality cutoff | Minimum quality per base in a sequence |
| Min percentage | Minimum percentage of bases in a sequence that meets the quality cutoff value |


In [4]:
from pathlib import Path
import gzip

input_file = Path("/Users/nguyuling/Galaxy/datasets/582600/mutant_R1.fastq")
output_file = Path("/Users/nguyuling/Galaxy/intro/filter/mutant_R1.filtered.fastq")

quality_cutoff = 35      # min phread quality per base
min_percentage = 80.0     # min percentage of bases meeting the cutoff

def open_fastq(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

total_reads = 0
kept_reads = 0

with open_fastq(input_file, "rt") as infile, open_fastq(output_file, "wt") as outfile:
    while True:
        header = infile.readline()
        if not header:
            break

        sequence = infile.readline()
        plus = infile.readline()
        quality = infile.readline()

        if not all((sequence, plus, quality)):
            raise ValueError("Incomplete FASTQ record found.")

        total_reads += 1

        quality_scores = [ord(base) - 33 for base in quality.strip()]
        passing_bases = sum(score >= quality_cutoff for score in quality_scores)
        passing_percentage = 100 * passing_bases / len(quality_scores)

        if passing_percentage >= min_percentage:
            outfile.write(header)
            outfile.write(sequence)
            outfile.write(plus)
            outfile.write(quality)
            kept_reads += 1

discarded_reads = total_reads - kept_reads
discarded_percentage = (
    100 * discarded_reads / total_reads if total_reads else 0
)

print(f"Input: {total_reads} reads")
print(f"Output: {kept_reads} reads")
print(
    f"discarded {discarded_reads} "
    f"({discarded_percentage:.2f}%) low-quality reads"
)
print(f"Filtered reads saved to: {output_file}")

Input: 12480 reads
Output: 10694 reads
discarded 1786 (14.31%) low-quality reads
Filtered reads saved to: /Users/nguyuling/Galaxy/intro/filter/mutant_R1.filtered.fastq
